<a href="https://colab.research.google.com/github/Evelyn-Rojas/Procesos-Estocasticos/blob/main/Metodo_Uniformacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<font color="blueviolet">**Evelyn Tania Rojas Roa**

---



#<font color="lightseagreen">**MÉTODO DE UNIFORMACIÓN**

---



Sea $R$ la matriz generadora de una CMTC. La matriz de probabilidades de transición $P(t)$ está dada por

$$
P(t)=e^{Rt}.
$$

Calcular directamente la exponencial matricial puede resultar costoso, por lo que se utiliza el **método de uniformización**, el cual transforma el problema en una suma infinita de potencias de una matriz estocástica.

Primero se elige una constante

$$
r \geq \max_i |r_{ii}|,
$$

y se define la matriz

$$
\hat{P}=I+\frac{R}{r}.
$$

Con esta definición, la matriz de transición puede escribirse como

$$
P(t)
=
\sum_{k=0}^{\infty}
e^{-rt}
\frac{(rt)^k}{k!}
\hat{P}^{\,k}.
$$

Esta expresión tiene una interpretación probabilística importante: los coeficientes

$$
e^{-rt}\frac{(rt)^k}{k!}
$$

corresponden a las probabilidades de una variable aleatoria de Poisson con parámetro $rt$.



En la práctica no es posible calcular una suma infinita, por lo que se utiliza una aproximación con los primeros $M$ términos:

$$
P^{(M)}(t)
=
\sum_{k=0}^{M}
e^{-rt}
\frac{(rt)^k}{k!}
\hat{P}^{\,k}.
$$

Mientras mayor sea el valor de $M$, mejor será la aproximación obtenida.

Una elección común es

$$
M \approx \max \{rt + 5\sqrt{rt},\,20\}.
$$


**Cota del error**

El error de truncamiento está dado por

$$
P(t)-P^{(M)}(t)
=
\sum_{k=M+1}^{\infty}
e^{-rt}
\frac{(rt)^k}{k!}
\hat{P}^{\,k}.
$$

Además, cada entrada de la matriz satisface

$$
\left|
p_{ij}(t)-p_{ij}^{(M)}(t)
\right|
\leq
\sum_{k=M+1}^{\infty}
e^{-rt}
\frac{(rt)^k}{k!}.
$$

Por lo tanto, para garantizar una tolerancia $\varepsilon$, basta elegir $M$ de forma que

$$
\sum_{k=M+1}^{\infty}
e^{-rt}
\frac{(rt)^k}{k!}
\leq
\varepsilon.
$$


<font color="mediumslateblue">**Algoritmo de uniformización**

1. Dados $R, t, 0 < \epsilon < 1$.
2. Calcular $r$ *usando la igualdad* en la definición.
3. Calcular $\hat{P}$.
4. $A = \hat{P}; \quad B = e^{-rt}I; \quad c = e^{-rt}; \quad sum = c; \quad k = 1$
5. Mientras $sum < 1 - \epsilon$ hacer:
   $$\begin{aligned}
   c &= c * (rt)/k \\
   B &= B + cA \\
   A &= A\hat{P} \\
   sum &= sum + c \\
   k &= k + 1
   \end{aligned}$$
6. $B$ está a $\epsilon$ de $P(t)$.

In [86]:
#Importamos las librerias
import numpy as np
from scipy.stats import poisson
from tabulate import tabulate #Mejora la vista al mostrar las matrices

**Ejercicio 1**

In [87]:
def Calcular_M(r,t):
  rt=r*t
  termino_var=rt+5*np.sqrt(rt)
  return int(np.ceil(max(termino_var,20))) #Tomamos solo la parte entera

**Ejercicio 2**

In [88]:
def Calcular_P_t(P_gorrito, r,t):
  num_estados= P_gorrito.shape[0]
  M= Calcular_M(r,t)
  P_t= np.zeros((num_estados,num_estados))
  P_gorrito_n= np.eye(num_estados)
  lamb= r*t
  for i in range(M+1):
    peso_poisson= poisson.pmf(i,lamb) #Función de masa de probabilidad
    P_t+= peso_poisson*P_gorrito_n
    P_gorrito_n=np.dot(P_gorrito_n,P_gorrito) #dot calcula el producto punto entre dos vectores
  return P_t,M

In [89]:
def Uniformizacion_Epsilon(P_gorrito, r, t, epsilon=0.00001):
    num_estados = P_gorrito.shape[0]
    rt = r * t
    # Inicialización según el algoritmo del teorema
    A = P_gorrito.copy()
    B = np.eye(num_estados) * np.exp(-rt)
    c = np.exp(-rt)
    sum_c = c
    k = 1
    while sum_c < 1 - epsilon:
        c = c * (rt) / k
        B = B + c * A
        A = np.dot(A, P_gorrito)
        sum_c = sum_c + c
        k = k + 1
    M_alcanzado = k - 1
    return B, M_alcanzado

**Definimos los parametros del ejercicio**

In [90]:
#Usando r=6
r=6
R = np.array([[0, 2, 3, 0], [4, 0, 2, 0],
    [0, 2, 0, 2], [1, 0, 3, 0]], dtype=float)
print("Matriz de tasas")
print(tabulate(R))

Matriz de tasas
-  -  -  -
0  2  3  0
4  0  2  0
0  2  0  2
1  0  3  0
-  -  -  -


In [91]:
def Calcular_P_gorrito(R, r):
    num_estados = R.shape[0]
    tasas_salida = np.sum(R, axis=1)
    P_gorrito = R / r
    diagonal_correcta = 1.0 - (tasas_salida / r)
    np.fill_diagonal(P_gorrito, diagonal_correcta)

    return P_gorrito

P_gorrito=Calcular_P_gorrito(R, r)
P_gorrito

array([[0.16666667, 0.33333333, 0.5       , 0.        ],
       [0.66666667, 0.        , 0.33333333, 0.        ],
       [0.        , 0.33333333, 0.33333333, 0.33333333],
       [0.16666667, 0.        , 0.5       , 0.33333333]])

In [92]:
tiempos = [0.5, 1.0, 5.0]
epsilon = 0.00001

In [93]:
resultados_m_fijo = {}
for t in tiempos:
    P_t, M = Calcular_P_t(P_gorrito, r, t)
    resultados_m_fijo[t] = P_t
    print("Resultados:")
    print(f"\n Para t = {t} (M = {M}) \n")
    print(P_t)


Resultados:

 Para t = 0.5 (M = 20) 

[[0.25060868 0.2169646  0.38665694 0.14576979]
 [0.25313484 0.23836098 0.37440924 0.13409493]
 [0.1691195  0.19361489 0.42030102 0.2169646 ]
 [0.15801748 0.15744464 0.39833179 0.28620609]]
Resultados:

 Para t = 1.0 (M = 20) 

[[0.20615112 0.20390203 0.3987096  0.1912358 ]
 [0.20828421 0.2053407  0.39789917 0.18847446]
 [0.19675849 0.19837934 0.40095869 0.20390203]
 [0.19204622 0.19399715 0.40147094 0.21248423]]
Resultados:

 Para t = 5.0 (M = 58) 

[[0.19999963 0.19999963 0.39999925 0.19999962]
 [0.19999963 0.19999963 0.39999925 0.19999962]
 [0.19999962 0.19999962 0.39999925 0.19999963]
 [0.19999962 0.19999962 0.39999925 0.19999963]]


In [94]:
print("VERIFICACIÓN DE CHAPMAN-KOLMOGOROV")
P_05 = resultados_m_fijo[0.5]
P_1 = resultados_m_fijo[1.0]
P_05_al_cuadrado = np.dot(P_05, P_05)

print("\nMatriz P(0.5) * P(0.5):")
print(P_05_al_cuadrado)
print("\nMatriz P(1):")
print(P_1)

VERIFICACIÓN DE CHAPMAN-KOLMOGOROV

Matriz P(0.5) * P(0.5):
[[0.20615141 0.20390232 0.39871018 0.19123609]
 [0.20828451 0.20534099 0.39789976 0.18847475]
 [0.19675878 0.19837963 0.40095927 0.20390232]
 [0.19204651 0.19399744 0.40147152 0.21248452]]

Matriz P(1):
[[0.20615112 0.20390203 0.3987096  0.1912358 ]
 [0.20828421 0.2053407  0.39789917 0.18847446]
 [0.19675849 0.19837934 0.40095869 0.20390203]
 [0.19204622 0.19399715 0.40147094 0.21248423]]


In [95]:
print("Algoritmo con tolerancia de epsilon")

for t in tiempos:
    P_t_eps, M_eps = Uniformizacion_Epsilon(P_gorrito, r, t, epsilon)
    M_fijo = Calcular_M(r, t)

    # Diferencia absoluta máxima entre ambos métodos
    diff_max = np.max(np.abs(resultados_m_fijo[t] - P_t_eps))

    print(f"\nPara t = {t} ")
    print(f"M (Fórmula fija): {M_fijo} | M (Con tolerancia): {M_eps}")
    print(f"Diferencia máxima entre matrices: {diff_max:.2e}")
    print("Matriz obtenida por tolerancia:")
    print(P_t_eps)

Algoritmo con tolerancia de epsilon

Para t = 0.5 
M (Fórmula fija): 20 | M (Con tolerancia): 13
Diferencia máxima entre matrices: 1.36e-06
Matriz obtenida por tolerancia:
[[0.250608   0.21696392 0.38665557 0.14576911]
 [0.25313416 0.2383603  0.37440788 0.13409425]
 [0.16911882 0.19361421 0.42029966 0.21696392]
 [0.1580168  0.15744396 0.39833043 0.28620541]]

Para t = 1.0 
M (Fórmula fija): 20 | M (Con tolerancia): 19
Diferencia máxima entre matrices: 1.49e-06
Matriz obtenida por tolerancia:
[[0.20615038 0.20390128 0.39870811 0.19123506]
 [0.20828347 0.20533995 0.39789768 0.18847371]
 [0.19675775 0.19837859 0.4009572  0.20390128]
 [0.19204548 0.1939964  0.40146945 0.21248349]]

Para t = 5.0 
M (Fórmula fija): 58 | M (Con tolerancia): 56
Diferencia máxima entre matrices: 2.20e-06
Matriz obtenida por tolerancia:
[[0.19999853 0.19999853 0.39999705 0.19999852]
 [0.19999853 0.19999853 0.39999705 0.19999852]
 [0.19999852 0.19999852 0.39999705 0.19999853]
 [0.19999852 0.19999852 0.39999705 0.